In [18]:
# check the machine

!nproc
!free -g | head -2
!pip list 2>/dev/null | grep -i -E "^(torch|transformers|safetensors|opencv)"
!ls /kaggle/input/datasets/saisdsaiiitbh/

4
               total        used        free      shared  buff/cache   available
Mem:              31           1          23           0           6          30
opencv-contrib-python                 4.14.0.94
opencv-python                         4.14.0.94
opencv-python-headless                4.14.0.94
safetensors                           0.8.0
torch                                 2.11.0+cu128
torchao                               0.10.0
torchaudio                            2.11.0+cu128
torchcodec                            0.11.0+cu128
torchdata                             0.11.0
torchinfo                             1.8.0
torchmetrics                          1.9.0
torchsummary                          1.5.1
torchtune                             0.6.1
torchvision                           0.26.0+cu128
transformers                          5.16.1
dff-code  eval-corps  ff-crops


In [ ]:
# Cross checking the mounting error

# from pathlib import Path
# import csv

# FF_ROOT = Path("/kaggle/input/datasets/saisdsaiiitbh/ff-crops")
# EV_ROOT = Path("/kaggle/input/datasets/saisdsaiiitbh/eval-corps")

# # Check the FF++ manifest
# manifest = FF_ROOT / "ffpp_manifest_split.csv"

# with manifest.open() as f:
#     rows = list(csv.DictReader(f))

# print("FF++ manifest rows:", len(rows))
# print("Manifest columns:", list(rows[0].keys()) if rows else "EMPTY")

# # Check a few frame directories
# for method in ["original", "Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]:
#     method_dir = FF_ROOT / "ffpp" / method
#     videos = list(method_dir.iterdir()) if method_dir.is_dir() else []

#     print(f"{method}: directory_exists={method_dir.is_dir()}, videos={len(videos)}")

# # Check evaluation manifests
# for name in ["cdf1_manifest.csv", "cdf2_manifest.csv"]:
#     p = EV_ROOT / name
#     print(f"{name}: exists={p.exists()}")

In [ ]:
# checking if smoke test is working fine or not


# import csv
# import random
# from pathlib import Path
# from collections import Counter

# FF_ROOT = Path("/kaggle/input/datasets/saisdsaiiitbh/ff-crops")
# SOURCE = FF_ROOT / "ffpp_manifest_split.csv"
# DEST = Path("/kaggle/working/ffpp_smoke_manifest.csv")

# random.seed(42)

# with SOURCE.open(newline="") as f:
#     rows = list(csv.DictReader(f))

# selected = []

# methods = [
#     "original",
#     "Deepfakes",
#     "Face2Face",
#     "FaceSwap",
#     "NeuralTextures",
# ]

# for method in methods:
#     method_rows = [r for r in rows if r["method"] == method]

#     train_rows = [r for r in method_rows if r["split"] == "train"]
#     val_rows = [r for r in method_rows if r["split"] == "val"]

#     assert len(train_rows) >= 40, f"Not enough training videos: {method}"
#     assert len(val_rows) >= 10, f"Not enough validation videos: {method}"

#     selected.extend(random.sample(train_rows, 40))
#     selected.extend(random.sample(val_rows, 10))

# with DEST.open("w", newline="") as f:
#     writer = csv.DictWriter(f, fieldnames=rows[0].keys())
#     writer.writeheader()
#     writer.writerows(selected)

# print("Smoke manifest:", DEST)
# print("Total videos:", len(selected))
# print("By method:", dict(Counter(r["method"] for r in selected)))
# print("By split:", dict(Counter(r["split"] for r in selected)))
# print("By method and split:")

# for method in methods:
#     print(
#         method,
#         "train =", sum(r["method"] == method and r["split"] == "train" for r in selected),
#         "val =", sum(r["method"] == method and r["split"] == "val" for r in selected),
#     )

In [ ]:
# %cd /kaggle/working
# !python train.py --help

In [ ]:
# !python test_dff_adapter.py --real

In [ ]:
# # a short benchmark run on real data. This runs one epoch with 1 random frame per video (3,600 samples):

# !python train.py --backbone small \
#   --crops_root /kaggle/input/datasets/saisdsaiiitbh/ff-crops \
#   --out_dir runs/bench --epochs 1 --frames_per_video 1 \
#   --batch_size 32 --num_workers 4 --val_frames 2 --log_every 10

In [ ]:
# # compute the speed.

# import json
# rows = [json.loads(l) for l in open("runs/bench/log.jsonl")]
# tr = [r for r in rows if "loss_auth" in r]
# print("steps:", tr[-1]["step"], "minutes:", tr[-1]["time_min"])
# print("samples/sec (both passes):", tr[-1]["step"]*32 / (tr[-1]["time_min"]*60))

In [ ]:
# import json
# rows = [json.loads(l) for l in open("runs/bench/log.jsonl")]
# tr = [r for r in rows if "loss_auth" in r]
# a, b = tr[0], tr[-1]
# print("first logged row at", a["time_min"], "min (everything before = listing + model load)")
# print("steady-state samples/s:", (b["step"]-a["step"])*32/((b["time_min"]-a["time_min"])*60))

In [ ]:
# # gpu speed alone

# import time, torch, torch.nn.functional as F
# from train import build_model
# m = build_model("small").cuda().train()
# opt = torch.optim.Adam(m.trainable_parameters(), 2e-4)
# scaler = torch.amp.GradScaler("cuda")
# x = torch.randn(32,3,224,224,device="cuda")
# ya = torch.randint(0,2,(32,),device="cuda").float()
# yt = torch.randint(0,5,(32,),device="cuda")
# def step():
#     with torch.autocast("cuda", dtype=torch.float16):
#         l = 10*F.binary_cross_entropy_with_logits(m(x,0).float(), ya) + 2*F.cross_entropy(m(x,1).float(), yt)
#     opt.zero_grad(); scaler.scale(l).backward(); scaler.step(opt); scaler.update()
# for _ in range(3): step()
# torch.cuda.synchronize(); t = time.time()
# for _ in range(20): step()
# torch.cuda.synchronize()
# print("GPU-only samples/s:", 20*32/(time.time()-t))

In [ ]:
# # data loading alone, no model

# import time
# from torch.utils.data import DataLoader
# from dataset import FrameDataset
# ROOT = "/kaggle/input/datasets/saisdsaiiitbh/ff-crops"
# t = time.time()
# ds = FrameDataset(ROOT, ROOT+"/ffpp_manifest_split.csv", "train", "train", 8)
# print("dataset init (listing folders): %.1fs" % (time.time()-t))
# for nw in (4, 8, 12):
#     it = iter(DataLoader(ds, batch_size=32, shuffle=True, num_workers=nw))
#     for _ in range(3): next(it)            # warm-up
#     t = time.time()
#     for _ in range(30): next(it)
#     print(f"{nw} workers: {30*32/(time.time()-t):.0f} samples/s (data only)")
#     del it

In [9]:
%cd /kaggle/working

!python -u train.py \
    --backbone small \
    --crops_root /kaggle/input/datasets/saisdsaiiitbh/ff-crops \
    --manifest /kaggle/working/ffpp_smoke_manifest.csv \
    --out_dir /kaggle/working/runs/smoke_250 \
    --epochs 2 \
    --frames_per_video 4 \
    --val_frames 4 \
    --batch_size 8 \
    --num_workers 2 \
    --log_every 5 \
    --ckpt_every 10 \
    --max_hours 0.5 \
    --seed 42

/kaggle/working
train: 200 videos -> 800 samples/epoch | val: 50 videos, 200 frames | device=cuda amp=True
config.json: 100%|█████████████████████████████| 547/547 [00:00<00:00, 2.41MB/s]

model.safetensors: downloading bytes: ███████████████████▋  | 79.1MB, 5.92MB/s  
model.safetensors: downloading bytes: ██████████████████████| 83.7MB, 8.04MB/s  
model.safetensors: reconstructing file: 100%|██████| 88.2MB / 88.2MB, 8.62MB/s  
Loading weights: 100%|██████████████████████| 223/223 [00:00<00:00, 2485.34it/s]
trainable parameters: 667,374
{'epoch': 0, 'step': 5, 'loss_auth': 1.2942115545272828, 'loss_type': 2.0353222846984864, 'acc_type': 0.175}
{'epoch': 0, 'step': 10, 'loss_auth': 0.7884491801261901, 'loss_type': 2.0800740718841553, 'acc_type': 0.25}
{'epoch': 0, 'step': 15, 'loss_auth': 0.6193182557821274, 'loss_type': 1.7717247247695922, 'acc_type': 0.2}
{'epoch': 0, 'step': 20, 'loss_auth': 1.019147366285324, 'loss_type': 1.8555307865142823, 'acc_type': 0.2}
{'epoch': 0, 'step': 25,

In [10]:
%cd /kaggle/working
!sed -n '1,240p' run_experiments.py

/kaggle/working
"""
run_experiments.py -- run named experiments (one per GPU, in parallel), evaluate each, summarise.

  python run_experiments.py dff_full dff_noaux        # train both, then evaluate both
  python run_experiments.py --summary                 # table of every finished run in runs/
  python run_experiments.py dff_full --dry_run        # just print the commands

Every experiment writes to runs/<name>/ (log.jsonl, last.pt, best.pt, done.json, results.json).
A run whose done.json exists is skipped for training, so re-running this script is safe.
best.pt = the epoch with the highest FF++ *validation* video AUC; it is evaluated on FF++ test and
on the unseen datasets CDF-v1 / CDF-v2 (test lists, all 32 frames per video).
"""
import argparse
import json
import os
import subprocess
import sys
import time
from pathlib import Path

FF_ROOT = os.environ.get("FF_ROOT", "/kaggle/input/datasets/saisdsaiiitbh/ff-crops")
EV_ROOT = os.environ.get("EV_ROOT", "/kaggle/input/datasets/sais

In [11]:
%cd /kaggle/working

from pathlib import Path

source = Path("run_experiments.py")
target = Path("run_experiments_smoke.py")

assert source.exists(), "run_experiments.py not found"
assert Path("ffpp_smoke_manifest.csv").exists(), "Smoke manifest not found"

script = source.read_text()

# Keep the original runner unchanged.
# Modify only the separate smoke-test copy.
script = script.replace(
    'FF_ROOT = os.environ.get("FF_ROOT", "/kaggle/input/datasets/saisdsaiiitbh/ff-crops")',
    'FF_ROOT = os.environ.get("FF_ROOT", "/kaggle/input/datasets/saisdsaiiitbh/ff-crops")\n'
    'SMOKE_MANIFEST = "/kaggle/working/ffpp_smoke_manifest.csv"'
)

script = script.replace(
    'COMMON = ["--backbone", "small", "--crops_root", FF_ROOT, "--epochs", "12", "--frames_per_video", "16",\n'
    '          "--val_frames", "8", "--batch_size", "24", "--num_workers", "8", "--ckpt_every", "500",\n'
    '          "--log_every", "100", "--max_hours", "5"]',
    'COMMON = ["--backbone", "small", "--crops_root", FF_ROOT,\n'
    '          "--manifest", SMOKE_MANIFEST, "--epochs", "2",\n'
    '          "--frames_per_video", "4", "--val_frames", "4",\n'
    '          "--batch_size", "8", "--num_workers", "2",\n'
    '          "--ckpt_every", "10", "--log_every", "5", "--max_hours", "0.5"]'
)

script = script.replace(
    'f"runs/{name}"',
    'f"runs/smoke_{name}"'
)

script = script.replace(
    'f"runs/{name}.log"',
    'f"runs/smoke_{name}.log"'
)

script = script.replace(
    'f"runs/{name}/done.json"',
    'f"runs/smoke_{name}/done.json"'
)

script = script.replace(
    'f"runs/{name}/best.pt"',
    'f"runs/smoke_{name}/best.pt"'
)

script = script.replace(
    'f"runs/{name}/results.json"',
    'f"runs/smoke_{name}/results.json"'
)

script = script.replace(
    'f"runs/{name}/eval.log"',
    'f"runs/smoke_{name}/eval.log"'
)

# Avoid mixing smoke-test results with full-experiment results.
script = script.replace(
    'Path("runs").glob("*/results.json")',
    'Path("runs").glob("smoke_*/results.json")'
)

target.write_text(script)

print("Created:", target)
print("Original runner unchanged:", source.exists())

/kaggle/working
Created: run_experiments_smoke.py
Original runner unchanged: True


In [12]:
%cd /kaggle/working
!python run_experiments_smoke.py dff_full dff_noaux --dry_run

/kaggle/working
[GPU 0] dff_full: /usr/bin/python3 train.py --backbone small --crops_root /kaggle/input/datasets/saisdsaiiitbh/ff-crops --manifest /kaggle/working/ffpp_smoke_manifest.csv --epochs 2 --frames_per_video 4 --val_frames 4 --batch_size 8 --num_workers 2 --ckpt_every 10 --log_every 5 --max_hours 0.5 --out_dir runs/smoke_dff_full
[GPU 1] dff_noaux: /usr/bin/python3 train.py --backbone small --crops_root /kaggle/input/datasets/saisdsaiiitbh/ff-crops --manifest /kaggle/working/ffpp_smoke_manifest.csv --epochs 2 --frames_per_video 4 --val_frames 4 --batch_size 8 --num_workers 2 --ckpt_every 10 --log_every 5 --max_hours 0.5 --out_dir runs/smoke_dff_noaux --no_aux
[GPU 0] eval dff_full: /usr/bin/python3 evaluate_dff.py --ckpt runs/smoke_dff_full/best.pt --backbone small --sets /kaggle/input/datasets/saisdsaiiitbh/ff-crops/ffpp_manifest_split.csv:test /kaggle/input/datasets/saisdsaiiitbh/eval-corps/cdf1_manifest.csv:all /kaggle/input/datasets/saisdsaiiitbh/eval-corps/cdf2_manifest.c

In [13]:
from pathlib import Path

p = Path("/kaggle/working/run_experiments_smoke.py")
script = p.read_text()

old = '''                print(f"{name}: training process exited with code {proc.returncode}", flush=True)
                evaluate(name, gpu, False)
                del running[gpu]'''

new = '''                print(f"{name}: training process exited with code {proc.returncode}", flush=True)

                if proc.returncode == 0 and Path(f"runs/smoke_{name}/done.json").exists():
                    print(f"{name}: training completed successfully.", flush=True)
                else:
                    print(f"{name}: training failed or did not complete.", flush=True)

                del running[gpu]'''

if old not in script:
    print("ERROR: Expected code was not found. File left unchanged.")
else:
    p.write_text(script.replace(old, new, 1))
    print("Updated run_experiments_smoke.py successfully.")

Updated run_experiments_smoke.py successfully.


In [14]:
!grep -n -A 10 'training process exited' /kaggle/working/run_experiments_smoke.py

136:                print(f"{name}: training process exited with code {proc.returncode}", flush=True)
137-
138-                if proc.returncode == 0 and Path(f"runs/smoke_{name}/done.json").exists():
139-                    print(f"{name}: training completed successfully.", flush=True)
140-                else:
141-                    print(f"{name}: training failed or did not complete.", flush=True)
142-
143-                del running[gpu]
144-    for n in args.names:                      # also evaluate runs that were already trained earlier
145-        if not args.dry_run and Path(f"runs/{n}/done.json").exists() and not Path(f"runs/{n}/results.json").exists():
146-            evaluate(n, 0, False)


In [15]:
%cd /kaggle/working
!python -u run_experiments_smoke.py dff_full dff_noaux

/kaggle/working
[GPU 0] dff_full: /usr/bin/python3 train.py --backbone small --crops_root /kaggle/input/datasets/saisdsaiiitbh/ff-crops --manifest /kaggle/working/ffpp_smoke_manifest.csv --epochs 2 --frames_per_video 4 --val_frames 4 --batch_size 8 --num_workers 2 --ckpt_every 10 --log_every 5 --max_hours 0.5 --out_dir runs/smoke_dff_full
[GPU 1] dff_noaux: /usr/bin/python3 train.py --backbone small --crops_root /kaggle/input/datasets/saisdsaiiitbh/ff-crops --manifest /kaggle/working/ffpp_smoke_manifest.csv --epochs 2 --frames_per_video 4 --val_frames 4 --batch_size 8 --num_workers 2 --ckpt_every 10 --log_every 5 --max_hours 0.5 --out_dir runs/smoke_dff_noaux --no_aux
dff_full: training process exited with code 0
dff_full: training completed successfully.
dff_noaux: training process exited with code 0
dff_noaux: training completed successfully.
run             FF++ AUC  CDF1 AUC  CDF2 AUC |  FF++ EER  CDF1 EER  CDF2 EER


In [19]:
from pathlib import Path

for name in ["dff_full", "dff_noaux"]:
    run_dir = Path(f"/kaggle/working/runs/smoke_{name}")
    print(f"\n--- {name} ---")

    for filename in ["best.pt", "last.pt", "done.json"]:
        path = run_dir / filename
        print(f"{filename}: {'EXISTS' if path.exists() else 'MISSING'}")

    log = Path(f"/kaggle/working/runs/smoke_{name}.log")
    if log.exists():
        print("\nLast 10 log lines:")
        print("\n".join(log.read_text(errors="replace").splitlines()[-10:]))


--- dff_full ---
best.pt: EXISTS
last.pt: EXISTS
done.json: EXISTS

Last 10 log lines:
{'epoch': 1, 'step': 165, 'loss_auth': 0.33864849209785464, 'loss_type': 1.2522452116012572, 'acc_type': 0.525}
{'epoch': 1, 'step': 170, 'loss_auth': 0.36669721603393557, 'loss_type': 1.2486536622047424, 'acc_type': 0.475}
{'epoch': 1, 'step': 175, 'loss_auth': 0.3491182684898376, 'loss_type': 1.3577762842178345, 'acc_type': 0.5}
{'epoch': 1, 'step': 180, 'loss_auth': 0.21861280500888824, 'loss_type': 1.2512067317962647, 'acc_type': 0.5}
{'epoch': 1, 'step': 185, 'loss_auth': 0.41171927452087403, 'loss_type': 1.268780255317688, 'acc_type': 0.5}
{'epoch': 1, 'step': 190, 'loss_auth': 0.21234080940485, 'loss_type': 1.2709017038345336, 'acc_type': 0.45}
{'epoch': 1, 'step': 195, 'loss_auth': 0.414588987827301, 'loss_type': 1.2294126510620118, 'acc_type': 0.425}
{'epoch': 1, 'step': 200, 'loss_auth': 0.3494760513305664, 'loss_type': 1.2551013708114624, 'acc_type': 0.575}
== epoch 1 validation: video AU

In [20]:
from pathlib import Path

for name in ["dff_full", "dff_noaux"]:
    run_dir = Path(f"/kaggle/working/runs/smoke_{name}")
    print(f"\n{name}")
    print("Results:", (run_dir / "results.json").exists())
    print("Evaluation log:", (run_dir / "eval.log").exists())


dff_full
Results: False
Evaluation log: False

dff_noaux
Results: False
Evaluation log: False


In [21]:
from pathlib import Path

for filename in ["run_experiments.py", "run_experiments_smoke.py"]:
    p = Path("/kaggle/working") / filename
    print(f"\n--- {filename} ---")

    for line in p.read_text().splitlines():
        if any(key in line for key in [
            "SMOKE_MANIFEST",
            '"--manifest"',
            '"--epochs"',
            '"--frames_per_video"',
            '"--batch_size"',
            '"--max_hours"'
        ]):
            print(line)


--- run_experiments.py ---
COMMON = ["--backbone", "small", "--crops_root", FF_ROOT, "--epochs", "12", "--frames_per_video", "16",
          "--val_frames", "8", "--batch_size", "24", "--num_workers", "8", "--ckpt_every", "500",
          "--log_every", "100", "--max_hours", "5"]

--- run_experiments_smoke.py ---
SMOKE_MANIFEST = "/kaggle/working/ffpp_smoke_manifest.csv"
          "--manifest", SMOKE_MANIFEST, "--epochs", "2",
          "--frames_per_video", "4", "--val_frames", "4",
          "--batch_size", "8", "--num_workers", "2",
          "--ckpt_every", "10", "--log_every", "5", "--max_hours", "0.5"]


In [22]:
import torch
import shutil
from pathlib import Path

print("Visible GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

usage = shutil.disk_usage("/kaggle/working")
print(f"\nFree disk space: {usage.free / (1024**3):.2f} GiB")

for name in ["dff_full", "dff_noaux"]:
    p = Path(f"/kaggle/working/runs/{name}")
    print(f"{name}: directory exists={p.exists()}, done.json exists={(p/'done.json').exists()}")

Visible GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

Free disk space: 19.46 GiB
dff_full: directory exists=False, done.json exists=False
dff_noaux: directory exists=False, done.json exists=False


## Actual running

In [23]:
%cd /kaggle/working
!python -u run_experiments.py dff_full dff_noaux

/kaggle/working
[GPU 0] dff_full: /usr/bin/python3 train.py --backbone small --crops_root /kaggle/input/datasets/saisdsaiiitbh/ff-crops --epochs 12 --frames_per_video 16 --val_frames 8 --batch_size 24 --num_workers 8 --ckpt_every 500 --log_every 100 --max_hours 5 --out_dir runs/dff_full
[GPU 1] dff_noaux: /usr/bin/python3 train.py --backbone small --crops_root /kaggle/input/datasets/saisdsaiiitbh/ff-crops --epochs 12 --frames_per_video 16 --val_frames 8 --batch_size 24 --num_workers 8 --ckpt_every 500 --log_every 100 --max_hours 5 --out_dir runs/dff_noaux --no_aux
[10:03:32] GPU0 dff_full: trainable parameters: 667,374
[10:03:32] GPU1 dff_noaux: trainable parameters: 667,374
[10:08:32] GPU0 dff_full: trainable parameters: 667,374
[10:08:32] GPU1 dff_noaux: trainable parameters: 667,374
[10:13:32] GPU0 dff_full: trainable parameters: 667,374
[10:13:32] GPU1 dff_noaux: trainable parameters: 667,374
^C
Traceback (most recent call last):
  File "/kaggle/working/run_experiments.py", line 14